# H&M M5 — arm B(N/V 분리 M4) 300 epoch 연장 (개발 seed 43)

10/04에 **epoch 100까지 학습해 판정이 끝난 같은 arm**을, Drive에 저장된 학습상태에서 **101 epoch부터 300까지 이어서** 돕니다. 수식·데이터·그래프·가중치·seed는 하나도 바뀌지 않고, 커밋도 그때와 같은 `7dcdf99`로 고정합니다(체크포인트 재개는 소스 리비전이 같아야 성립합니다).

**왜 연장하나.** 100 epoch 판정의 비교 상대였던 M4는 100 epoch 기록만 있었습니다. 별도 노트북에서 **M4를 같은 학습 루프로 300 epoch까지** 돌리고 있어, 세 모형을 같은 epoch에서 읽기 위해 A·B도 300까지 올립니다.

**표기 — 반드시 지킬 것.** epoch 100 판정은 사전등록된 판정으로 **그대로 남습니다.** 300은 사전등록된 **연장 확인**이며, 결과 방향과 무관하게 보고하고 100 판정을 대체하지 않습니다. 300 지점에는 순열 arm이 없으므로 그 지점의 수치를 CLV 배정 효과로 귀속할 수 없습니다.

고정: H&M 개발분할(2020-09-02~08), seed 43, 300 epoch 고정 종료. 얼리스탑핑·epoch 선택 없음. test/holdout은 열지 않습니다.

**런타임:** GPU + **고용량 RAM**. 남은 200 epoch에 약 25시간(기록 epoch당 약 450초). 끊기면 같은 셀을 다시 실행하면 마지막 epoch부터 이어집니다. 100·200·300 epoch에서 평가가 찍힙니다.

In [ ]:
from pathlib import Path
import json, os, subprocess, sys
from google.colab import drive

ROOT = Path('/content/drive/MyDrive/논문/data')
REFS = [ROOT/'results_v3_hm_clv_m3_centered_value_graph_s43_v1', ROOT/'results_v3_hm_m4_k1_assignment_control_hm2y_development_screen_v1']
if not os.path.ismount('/content/drive'):
    try:
        drive.mount('/content/drive')
    except (ValueError, NotImplementedError) as exc:
        raise RuntimeError('Drive 연결 실패. 학습을 시작하지 않았습니다.') from exc
for ref in REFS:
    if not list(ref.glob('*.json')):
        raise RuntimeError(f'기준 결과가 없습니다: {ref} — 재학습하지 않고 중단합니다.')

SOURCE_COMMIT = '7dcdf99e5924b93545c231dec82fb900f7e453b1'
VARIANT = 'split_nv'   # arm B
REPO = Path('/content/clv-m5-hm2y-' + SOURCE_COMMIT[:12])
if not REPO.exists():
    subprocess.run(['git', 'clone', 'https://github.com/jung-un/clv-m2-lightgcn-runner.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '--detach', SOURCE_COMMIT], check=True)
assert subprocess.check_output(['git', '-C', str(REPO), 'rev-parse', 'HEAD'], text=True).strip() == SOURCE_COMMIT
for name in ('lightgcn_clv_v3', 'lightgcn_clv_m3_centered_value_graph_hm2y', 'clv_m5_m3_m4_split_nv_hm2y_screen'):
    if name in sys.modules and Path(sys.modules[name].__file__).resolve().parent != REPO.resolve():
        raise RuntimeError(f'{name}이 다른 코드에서 로드됐습니다. 런타임을 다시 시작하세요.')
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import torch
assert torch.cuda.is_available(), '학습에는 GPU 런타임을 사용하세요.'
import clv_m5_m3_m4_split_nv_screen
clv_m5_m3_m4_split_nv_screen.self_test()
import clv_m5_m3_m4_split_nv_hm2y_screen as screen
cfg = screen.configure()
print(json.dumps({'source_commit': SOURCE_COMMIT, 'arm': VARIANT, 'model_id': screen.ARMS[VARIANT],
                  'judge_epoch': screen.JUDGE_EPOCH, 'out_dir': cfg.out_dir}, ensure_ascii=False, indent=2))

## 데이터 준비 → 학습 전 점검 → 학습

준비 단계에서 ① 그래프 강도 β가 H&M M3 기준(0.6088)과 같은지 ② M1·M3·M4 기준 결과가 같은 설정인지, 두 실행의 M1@100이 같은지 ③ A의 가중치가 확증된 H&M M4 감사값(평균 원시가중 1.21998, CV 0.15905)과 같은지 확인하고, 하나라도 다르면 학습하지 않고 멈춥니다.

In [ ]:
prepared = screen.prepare(cfg)
arm = screen.train_arm(cfg, prepared, VARIANT, stop_epoch=300)

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from google.colab import files
result = screen.report(cfg, prepared)   # 끝난 arm만 비교 (다른 arm이 아직이면 이 arm만)
key = ['recall@10', 'ndcg@10', 'recall@20', 'ndcg@20', 'recall@50', 'ndcg@50',
       'price_purchase_amount_weighted_hit@10', 'vndcg@10',
       'price_purchase_amount_weighted_hit@20', 'vndcg@20',
       'price_purchase_amount_weighted_hit@50', 'vndcg@50']
view = result['comparison']
view = view[view.metric.isin(key)]
for (epoch, model, ref), part in view.groupby(['epoch', 'model_id', 'reference']):
    print(f'\n=== epoch {epoch} | {model} vs {ref} ===')
    print(part[['metric', 'reference_value', 'candidate_value', 'ratio']].to_string(index=False))
print(json.dumps(result['reading'], ensure_ascii=False, indent=2))
zip_path = Path('/content/clv_m5_m3_m4_split_nv_hm2y_s43_split_nv_to300_results.zip')
with ZipFile(zip_path, 'w', compression=ZIP_DEFLATED) as archive:
    for path in result['paths'].values():
        archive.write(path, arcname=Path(path).name)
files.download(str(zip_path))